# vessel-watch: train on Colab

Launcher only. All logic lives in the `vesselwatch` package; this notebook mounts Drive, fetches the repo, copies the dataset to fast local disk and starts a run.

Runtime > Change runtime type > GPU before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'  # set once the GitHub repo exists
BRANCH = 'main'
CONFIG = 'configs/legacy/level1_net.yaml'

DATA_DIR_DRIVE = '/content/drive/MyDrive/5.Projects/Data/ShipRSImageNet_V1'
DATA_DIR_LOCAL = '/content/data/ShipRSImageNet_V1'
RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'  # results are written straight to Drive

## Code

In [ ]:
import os, subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn and pyyaml
%pip install -q --no-deps -e /content/vessel-watch
%cd /content/vessel-watch
!git log --oneline -1

## Data

Reading images from Drive during training is slow, so they are copied to the Colab disk once per session.

In [ ]:
import shutil

if os.path.exists(DATA_DIR_LOCAL):
    print('Local copy already exists')
else:
    os.makedirs(os.path.dirname(DATA_DIR_LOCAL), exist_ok=True)
    shutil.copytree(DATA_DIR_DRIVE, DATA_DIR_LOCAL)
    print('Copied dataset to', DATA_DIR_LOCAL)

os.environ['VESSELWATCH_DATA'] = DATA_DIR_LOCAL
os.environ['VESSELWATCH_RUNS'] = RUNS_DIR

## Train

In [ ]:
!python -m vesselwatch.train --config $CONFIG

## Evaluate saved weights (optional)

Works with checkpoints from this package and with the `.pth` files of the original study.

In [ ]:
WEIGHTS = f"{RUNS_DIR}/{os.path.splitext(os.path.basename(CONFIG))[0]}/best.pt"
!python -m vesselwatch.evaluate --config $CONFIG --weights "$WEIGHTS"